In [2]:
import pandas as pd
import sys
from datetime import datetime
from os import environ
from platform import system
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import json


load_dotenv()

DATA_DIR = Path("../_data")
OUTPUT_DIR = Path("../_output")
OPENAI_API_KEY = environ.get("OPENAI_API_KEY")
METADATA_JSON = OUTPUT_DIR / "openai_headline_batch_metadata.json"
BATCH_OUTPUT_JSONL = OUTPUT_DIR / "openai_headline_batch_output.jsonl"
BATCH_ERROR_JSONL = OUTPUT_DIR / "openai_headline_batch_errors.jsonl"

In [3]:
print(DATA_DIR)
print(OUTPUT_DIR)
print(f"{DATA_DIR}/RAVENPACK_cleaned.parquet")

../_data
../_output
../_data/RAVENPACK_cleaned.parquet


In [15]:
df = pd.read_parquet(f"{DATA_DIR}/RAVENPACK_cleaned.parquet")

In [43]:
import math
total = df.index.size
chunk_size = 40000


In [ ]:
prev_chunk_end = 0
chunks = []
while total > 0:
    print("total to split: ", total)
    chunk = (prev_chunk_end, prev_chunk_end + min(chunk_size, total+1))
    chunks.append(chunk)
    prev_chunk_end += min(chunk_size, total+1)
    total -= chunk_size

chunks

total to split:  110341
total to split:  70341
total to split:  30341


[(0, 40000), (40000, 80000), (80000, 110341)]

In [46]:
print(df.iloc[0:10])
print(df.iloc[10:20])
print(df.iloc[20:30])

  rp_entity_id rpa_date_utc           timestamp_utc map_ticker  \
0       000AD2   2023-09-11 2023-09-11 21:25:02.641       DTSS   
1       000AD2   2022-03-10 2022-03-10 13:30:02.488       DTSS   
2       000AD2   2023-06-13 2023-06-13 12:30:00.722       DTSS   
3       000AD2   2022-05-23 2022-05-23 12:30:01.507       DTSS   
4       000AD2   2023-09-12 2023-09-12 12:30:09.882       DTSS   
5       000AD2   2023-09-13 2023-09-13 20:05:06.173       DTSS   
6       001F1B   2024-01-25 2024-01-25 23:49:49.766       PSMT   
7       001F1B   2022-07-05 2022-07-05 12:00:13.043       PSMT   
8       001F1B   2023-11-18 2023-11-18 00:15:46.847       PSMT   
9       001F1B   2022-12-16 2022-12-16 13:00:04.059       PSMT   

       entity_name                                           headline  
0     Datasea Inc.  Datasea Announces Proposed Underwritten Public...  
1     Datasea Inc.  Datasea Launches New Product and Deepens Coope...  
2     Datasea Inc.  Datasea Joins Forces with Industry Le

In [12]:
df.iloc[500:501]

,rp_entity_id,rpa_date_utc,timestamp_utc,map_ticker,entity_name,headline
500,05AF8F,2021-11-24,2021-11-24 08:32:35.577,GNS,Genus PLC,Genus Expects Lower FY 2022 Pretax Profit on C...


In [79]:
import submit_headlines_to_openai as open_ai_adapter
import process_openai_responses as open_ai_processor
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# result = open_ai_adapter.make_requests_jsonl(sub_df, "gpt-4")
# result

In [79]:
batch_file_id = open_ai_adapter._upload_batch_file(openai_client)

Uploaded batch file: file-8jD16j49SGdUcUbYmwt2aF


In [80]:
batch_id = open_ai_adapter._create_batch_job(openai_client, batch_file_id)

Created batch job: batch_6995ecf7e7a48190a05f21d5dca09add


In [81]:
batch_data = open_ai_adapter._poll_until_done(openai_client, batch_id)

Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status

In [82]:
batch_data

Batch(id='batch_6995ecf7e7a48190a05f21d5dca09add', completion_window='24h', created_at=1771433207, endpoint='/v1/chat/completions', input_file_id='file-8jD16j49SGdUcUbYmwt2aF', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1771434323, error_file_id=None, errors=None, expired_at=None, expires_at=1771519607, failed_at=None, finalizing_at=1771434321, in_progress_at=1771433209, metadata={'job_name': 'ravenpack_headline_scoring'}, model='gpt-4-0613', output_file_id='file-Lm821p9t7K547dxg1tBpXd', request_counts=BatchRequestCounts(completed=10, failed=0, total=10), usage=BatchUsage(input_tokens=1127, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=264, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1391))

In [83]:
batch_data.model_dump()

{'id': 'batch_6995ecf7e7a48190a05f21d5dca09add',
 'completion_window': '24h',
 'created_at': 1771433207,
 'endpoint': '/v1/chat/completions',
 'input_file_id': 'file-8jD16j49SGdUcUbYmwt2aF',
 'object': 'batch',
 'status': 'completed',
 'cancelled_at': None,
 'cancelling_at': None,
 'completed_at': 1771434323,
 'error_file_id': None,
 'errors': None,
 'expired_at': None,
 'expires_at': 1771519607,
 'failed_at': None,
 'finalizing_at': 1771434321,
 'in_progress_at': 1771433209,
 'metadata': {'job_name': 'ravenpack_headline_scoring'},
 'model': 'gpt-4-0613',
 'output_file_id': 'file-Lm821p9t7K547dxg1tBpXd',
 'request_counts': {'completed': 10, 'failed': 0, 'total': 10},
 'usage': {'input_tokens': 1127,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 264,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 1391}}

In [84]:
METADATA_JSON.parent.mkdir(parents=True, exist_ok=True)
METADATA_JSON.write_text(
    json.dumps(batch_data.model_dump(), indent=2, default=str),
    encoding="utf-8",
)
print(f"Saved batch metadata to: {METADATA_JSON}")

Saved batch metadata to: ../_output/openai_headline_batch_metadata.json


In [85]:
output_file_id = batch_data.output_file_id
error_file_id = batch_data.error_file_id

In [87]:
open_ai_adapter._download_file_content(openai_client, output_file_id, BATCH_OUTPUT_JSONL)

Saved file content to: ../_output/openai_headline_batch_output.jsonl


In [53]:
open_ai_adapter._download_file_content(openai_client, error_file_id, BATCH_ERROR_JSONL)

ValueError: Expected a non-empty value for `file_id` but received None

In [14]:
# columns = ['entity_name', 'map_ticker', 'timestamp_utc', 'rpa_date_utc', 'timestamp_et', 'date', 'headline']
columns = ['timestamp_utc', 'rpa_date_utc', 'timestamp_et', 'date']
from zoneinfo import ZoneInfo

df = df.copy()
ts = pd.to_datetime(df['timestamp_utc'], errors='coerce')
if ts.dt.tz is None:
    ts = ts.dt.tz_localize('UTC')

df.loc[:, 'timestamp_et'] = ts.dt.tz_convert(ZoneInfo('America/New_York'))
df['date'] = df['timestamp_et'].dt.date
df['rpa_date_utc'] = pd.to_datetime(df['rpa_date_utc'], errors='coerce').dt.date
df['date_eq'] = df['date'] == df['rpa_date_utc']
df.loc[~df['date_eq']][columns]
# df.loc[:, 'date'] = ts.dt.tz_convert(ZoneInfo('America/New_York')).dt.date
# df = df.loc[:, columns].copy()

,timestamp_utc,rpa_date_utc,timestamp_et,date
16,2023-11-18 00:15:46.847,2023-11-18,2023-11-17 19:15:46.847000-05:00,2023-11-17
21,2023-03-30 01:22:47.685,2023-03-30,2023-03-29 21:22:47.685000-04:00,2023-03-29
45,2023-10-24 01:33:27.257,2023-10-24,2023-10-23 21:33:27.257000-04:00,2023-10-23
59,2022-03-04 00:16:24.939,2022-03-04,2022-03-03 19:16:24.939000-05:00,2022-03-03
95,2022-03-18 00:43:10.244,2022-03-18,2022-03-17 20:43:10.244000-04:00,2022-03-17
...,...,...,...,...
135706,2023-02-09 00:07:13.723,2023-02-09,2023-02-08 19:07:13.723000-05:00,2023-02-08
135722,2024-03-12 00:41:48.095,2024-03-12,2024-03-11 20:41:48.095000-04:00,2024-03-11
135736,2024-01-23 00:13:24.637,2024-01-23,2024-01-22 19:13:24.637000-05:00,2024-01-22
135748,2022-02-11 02:07:19.233,2022-02-11,2022-02-10 21:07:19.233000-05:00,2022-02-10


In [ ]:
import json
import os
import re
import time
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
from openai import OpenAI
from settings import config

DATA_DIR = Path(config("DATA_DIR"))
OUTPUT_DIR = Path(config("OUTPUT_DIR"))
OPENAI_API_KEY = config("OPENAI_API_KEY")
OPENAI_MODEL = config("OPENAI_MODEL")

INPUT_CANDIDATE = DATA_DIR / "RAVENPACK_cleaned.parquet"
REQUESTS_JSONL = DATA_DIR / "openai_headline_requests.jsonl"
SCORES_PARQUET = DATA_DIR / "daily_headline_polarity.parquet"

BATCH_OUTPUT_JSONL = OUTPUT_DIR / "openai_headline_batch_output.jsonl"
BATCH_ERROR_JSONL = OUTPUT_DIR / "openai_headline_batch_errors.jsonl"
METADATA_JSON = OUTPUT_DIR / "openai_headline_batch_metadata.json"
ID_ROW_JSON = OUTPUT_DIR / "id_to_row_mapping.json"

In [ ]:
id_to_row = pd.read_json(ID_ROW_JSON, orient="index")


In [29]:
id_to_row

,ticker,date,entity_name
rp-0,HTGC,2022-01-14,Hercules Capital Inc.
rp-1,HTGC,2022-08-30,Hercules Capital Inc.
rp-2,HTGC,2023-08-07,Hercules Capital Inc.
rp-3,HTGC,2023-04-20,Hercules Capital Inc.
rp-4,HTGC,2024-02-15,Hercules Capital Inc.
rp-5,HTGC,2022-02-22,Hercules Capital Inc.
rp-6,HTGC,2024-01-23,Hercules Capital Inc.
rp-7,HTGC,2023-07-10,Hercules Capital Inc.
rp-8,HTGC,2024-04-18,Hercules Capital Inc.
rp-9,HTGC,2022-05-05,Hercules Capital Inc.


In [68]:
id_to_row.loc['rp-11'] = id_to_row.loc['rp-6'].copy()

In [69]:
id_to_row

,ticker,date,entity_name
rp-0,HTGC,2022-01-14,Hercules Capital Inc.
rp-1,HTGC,2022-08-30,Hercules Capital Inc.
rp-2,HTGC,2023-08-07,Hercules Capital Inc.
rp-3,HTGC,2023-04-20,Hercules Capital Inc.
rp-4,HTGC,2024-02-15,Hercules Capital Inc.
rp-5,HTGC,2022-02-22,Hercules Capital Inc.
rp-6,HTGC,2024-01-23,Hercules Capital Inc.
rp-7,HTGC,2023-07-10,Hercules Capital Inc.
rp-8,HTGC,2024-04-18,Hercules Capital Inc.
rp-9,HTGC,2022-05-05,Hercules Capital Inc.


In [80]:
open_ai_processor.build_scores_df(id_to_row)

,ticker,date,n_headlines,score_sum
0,HTGC,2022-01-14,1,0
1,HTGC,2022-02-22,1,1
2,HTGC,2022-05-05,1,0
3,HTGC,2022-08-30,1,1
4,HTGC,2023-04-20,1,0
5,HTGC,2023-07-10,1,0
6,HTGC,2023-08-07,1,1
7,HTGC,2024-01-23,1,1
8,HTGC,2024-02-15,1,0
9,HTGC,2024-04-18,1,0


In [81]:
df = pd.read_parquet(SCORES_PARQUET)
df.head()

,ticker,date,n_headlines,score_sum
0,HTGC,2022-01-14,1,0
1,HTGC,2022-02-22,1,1
2,HTGC,2022-05-05,1,0
3,HTGC,2022-08-30,1,1
4,HTGC,2023-04-20,1,0
